# 09 · TensorFlow Lite Deployment: FP32, FP16 and INT8 with On-Device Heatmaps

**Goal (proposal objective 5):** make DFU-LiteNet phone-ready and measure what that costs.

For each trained model (Baseline and the selected attention variant, all 5 folds) this notebook:
1. Builds a **deployment model with two outputs**: the ulcer probability **and** the 7×7 heatmap.
   Because the head is global average pooling + one dense neuron, the heatmap is simply the last feature map weighted by that neuron's
   weights (the CAM form of Grad-CAM, notebook 08). No gradients are needed, so a phone can draw it at almost no extra cost.
2. Converts it to **TensorFlow Lite** in three precisions:
   - **FP32**: no compression (the reference);
   - **FP16**: weights stored as 16-bit floats (about half the size);
   - **INT8-dyn**: dynamic-range quantisation, int8 weights with float maths (about 4x smaller, no calibration);
   - **INT8**: full-integer quantisation of weights and activations, calibrated on **training images only** (never test images).
3. Measures, for each version: **file size**, **test accuracy/F1/AUC** (each fold's own test images and threshold),
   **agreement** with the original Keras model, **heatmap fidelity** (correlation with the Keras heatmap), and **CPU latency**.

The fold-0 files are saved under `tflite/` for the Android test (notebook 10 / phone app).

**Inputs:** the dataset, **DFU 03 - Leakage-Free Split** and **DFU 06 - LiteNet Ablation**.
**Accelerator:** None (CPU) is fine; latency is measured on CPU · **Internet:** On · **Time:** about 20-40 minutes.

In [ ]:
import json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

CONFIG = {
    "FOLDS": [0, 1, 2, 3, 4],
    "IMG_SIZE": 224,
    "N_CALIBRATION": 200,     # training images used to calibrate INT8 (never test images)
    "LATENCY_RUNS": 100,
    "THREADS": [1, 4],        # 1 thread = a weak phone core; 4 threads = a typical phone using several cores
    "SEED": 42,
}
S = CONFIG["IMG_SIZE"]
INPUT = Path("/kaggle/input")
PATCHES = next(INPUT.rglob("Patches"))
SPLITS = next(INPUT.rglob("splits.csv"))
WEIGHTS = next(INPUT.rglob("Baseline_fold0.npz")).parent
OUT06 = WEIGHTS.parent
SELECTED = json.load(open(OUT06 / "selection.json"))["selected_attention"]
VARIANTS = ["Baseline", SELECTED]
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("Variants:", VARIANTS, "| weights from", WEIGHTS)

splits = pd.read_csv(SPLITS)
kd = splits[splits.keep].reset_index(drop=True)
X = np.stack([np.asarray(Image.open(PATCHES / r["class"] / r.file).convert("RGB").resize((S, S), Image.BILINEAR), dtype=np.float32)
              for _, r in kd.iterrows()])
y = kd.label.values
print("Images:", X.shape)

## 1. Rebuild the notebook 06 models and add the heatmap output
Same model code as notebooks 06 and 08. The deployment model returns `[probability, heatmap 7×7]`.

In [ ]:
class ECA(layers.Layer):
    def __init__(self, gamma=2, b=1, **kw):
        super().__init__(**kw); self.gamma, self.b = gamma, b
    def build(self, shape):
        t = int(abs((np.log2(shape[-1]) + self.b) / self.gamma))
        k = t if t % 2 else t + 1
        self.conv = layers.Conv1D(1, k, padding="same", use_bias=False, kernel_initializer="zeros")
        self.conv.build((None, shape[-1], 1))
    def call(self, x):
        z = ops.expand_dims(ops.mean(x, axis=(1, 2)), -1)
        gate = 2.0 * ops.sigmoid(self.conv(z))
        return x * ops.reshape(gate, (-1, 1, 1, x.shape[-1]))

class CoordAtt(layers.Layer):
    def __init__(self, reduction=32, **kw):
        super().__init__(**kw); self.reduction = reduction
    def build(self, shape):
        c = shape[-1]; mip = max(8, c // self.reduction)
        self.conv1 = layers.Conv2D(mip, 1, use_bias=False)
        self.bn = layers.BatchNormalization(momentum=0.9)
        self.conv_h = layers.Conv2D(c, 1, kernel_initializer="zeros")
        self.conv_w = layers.Conv2D(c, 1, kernel_initializer="zeros")
        self.conv1.build((None, None, 1, c)); self.bn.build((None, None, 1, mip))
        self.conv_h.build((None, None, 1, mip)); self.conv_w.build((None, 1, None, mip))
    def call(self, x, training=None):
        h, w = x.shape[1], x.shape[2]
        x_h = ops.mean(x, axis=2, keepdims=True)
        x_w = ops.transpose(ops.mean(x, axis=1, keepdims=True), (0, 2, 1, 3))
        y_ = ops.relu(self.bn(self.conv1(ops.concatenate([x_h, x_w], axis=1)), training=training))
        a_h = 2.0 * ops.sigmoid(self.conv_h(y_[:, :h]))
        a_w = 2.0 * ops.sigmoid(self.conv_w(ops.transpose(y_[:, h:], (0, 2, 1, 3))))
        return x * a_h * a_w

ATT = {"ECA": ECA, "CA": CoordAtt}
STAGE_ENDS = ["expanded_conv_2_add", "expanded_conv_7_add"]

def build_model(variant):
    base = keras.applications.MobileNetV3Small(include_top=False, weights=None, input_shape=(S, S, 3), include_preprocessing=True)
    kind, _, where = variant.partition("-")
    net = base
    inside = {"Multi": STAGE_ENDS, "Early": STAGE_ENDS[:1]}.get(where, [])
    if inside:
        def call_fn(layer, *args, **kwargs):
            out = layer(*args, **kwargs)
            return ATT[kind](name=f"att_{layer.name}")(out) if layer.name in inside else out
        net = keras.models.clone_model(base, clone_function=lambda l: l, call_function=call_fn)
    x = net.output
    if where in ("Late", "Multi"):
        x = ATT[kind](name="att_final")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="classifier")(x)
    return keras.Model(net.input, out, name=variant.replace("-", "_"))

def load_deploy(variant, fold):
    model = build_model(variant)
    f = np.load(WEIGHTS / f"{variant}_fold{fold}.npz")
    n = len(model.get_weights())
    assert len(f.files) - 1 == n, f"{variant} fold {fold}: weight count mismatch"
    model.set_weights([f[f"w{i}"] for i in range(n)])
    w = model.get_layer("classifier").get_weights()[0][:, 0]
    feats = model.get_layer("gap").input
    # heatmap = ReLU(sum_k w_k A_k), written as a fixed 1x1 convolution so every TFLite backend (incl. INT8 on phones) supports it
    cam_conv = layers.Conv2D(1, 1, use_bias=False, name="cam_weights")
    cam = layers.ReLU(name="cam")(cam_conv(feats))
    cam_conv.set_weights([w.reshape(1, 1, -1, 1)])
    deploy = keras.Model(model.input, [model.output, cam], name=model.name + "_deploy")
    return deploy, float(f["threshold"])

## 2. Convert, test and time every version
INT8 calibration uses images from the **training folds** of each run only. Test images are never seen during conversion.

In [ ]:
def convert(deploy, mode, calib):
    conv = tf.lite.TFLiteConverter.from_keras_model(deploy)   # (from_concrete_functions gave NaN outputs with Keras 3)
    if mode == "INT8-dyn":                     # dynamic range: int8 weights, float activations (no calibration needed)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    elif mode == "FP16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    elif mode == "INT8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = lambda: ([calib[i:i + 1]] for i in range(len(calib)))
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # input and output stay float32 so the app code is the same for every version
    return conv.convert()

NO_XNN = tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES

def interpreter(blob, threads):
    # XNNPACK (the default CPU accelerator, also on Android) first; if it rejects an op, the built-in TFLite kernels
    try:
        it = tf.lite.Interpreter(model_content=blob, num_threads=threads); it.allocate_tensors(); return it, "XNNPACK"
    except RuntimeError:
        it = tf.lite.Interpreter(model_content=blob, num_threads=threads, experimental_op_resolver_type=NO_XNN)
        it.allocate_tensors(); return it, "built-in"

def run_tflite(blob, Xs, threads=4):
    it, _ = interpreter(blob, threads)
    inp = it.get_input_details()[0]["index"]
    outs = sorted(it.get_output_details(), key=lambda d: len(d["shape"]))   # probability (rank 2) first, heatmap (rank 4) second
    probs, cams = [], []
    for x in Xs:
        it.set_tensor(inp, x[None])
        it.invoke()
        probs.append(float(it.get_tensor(outs[0]["index"]).ravel()[0])); cams.append(it.get_tensor(outs[1]["index"])[0, ..., 0])
    return np.array(probs), np.stack(cams)

def latency(blob, threads, runs):
    it, _ = interpreter(blob, threads)
    inp = it.get_input_details()[0]["index"]
    x = X[:1]
    for _ in range(10):
        it.set_tensor(inp, x); it.invoke()
    t = time.perf_counter()
    for _ in range(runs):
        it.set_tensor(inp, x); it.invoke()
    return (time.perf_counter() - t) / runs * 1000

def cam_corr(a, b):
    a, b = a.reshape(len(a), -1), b.reshape(len(b), -1)
    ok = (a.std(1) > 0) & (b.std(1) > 0)
    return float(np.mean([np.corrcoef(a[i], b[i])[0, 1] for i in np.where(ok)[0]])) if ok.any() else np.nan

MODES = ["FP32", "FP16", "INT8-dyn", "INT8"]
os.makedirs("tflite", exist_ok=True)
rows = []
rng = np.random.default_rng(CONFIG["SEED"])
t0 = time.time()
for v in VARIANTS:
    for k in CONFIG["FOLDS"]:
        deploy, thr = load_deploy(v, k)
        val = (k + 1) % 5
        te = np.where(kd.fold == k)[0]
        tr = np.where((kd.fold != k) & (kd.fold != val))[0]
        calib = X[rng.choice(tr, min(CONFIG["N_CALIBRATION"], len(tr)), replace=False)]
        p_ref, cam_ref = deploy.predict(X[te], batch_size=64, verbose=0)
        p_ref, cam_ref = p_ref.ravel(), cam_ref[..., 0]
        for mode in MODES:
            blob = convert(deploy, mode, calib)
            p, cam = run_tflite(blob, X[te])
            pred = (p >= thr).astype(int)
            row = {"variant": v, "fold": k, "precision": mode, "size_mb": len(blob) / 1e6,
                   "accuracy": accuracy_score(y[te], pred), "f1": f1_score(y[te], pred), "auc": roc_auc_score(y[te], p),
                   "missed_ulcers": int(((y[te] == 1) & (pred == 0)).sum()), "false_alarms": int(((y[te] == 0) & (pred == 1)).sum()),
                   "label_agreement_vs_keras": float((pred == (p_ref >= thr)).mean()),
                   "max_prob_diff_vs_keras": float(np.abs(p - p_ref).max()),
                   "heatmap_corr_vs_keras": cam_corr(cam, cam_ref), "cpu_backend": interpreter(blob, 1)[1]}
            if k == CONFIG["FOLDS"][0]:          # latency once per variant and precision (the fold does not change the graph)
                for th in CONFIG["THREADS"]:
                    row[f"cpu_ms_{th}thread"] = latency(blob, th, CONFIG["LATENCY_RUNS"])
                open(f"tflite/{v}_fold{k}_{mode}.tflite", "wb").write(blob)
            rows.append(row)
            print(f"{v:10s} fold {k} {mode} | {row['size_mb']:.2f} MB | F1 {row['f1']:.3f} | agreement {row['label_agreement_vs_keras']:.3f}"
                  f" | heatmap corr {row['heatmap_corr_vs_keras']:.3f} | {time.time() - t0:.0f}s", flush=True)
        keras.backend.clear_session()

res = pd.DataFrame(rows)
res.to_csv("tflite_results_per_fold.csv", index=False)
json.dump({v: float(np.load(WEIGHTS / f"{v}_fold0.npz")["threshold"]) for v in VARIANTS}, open("tflite/thresholds_fold0.json", "w"), indent=2)
print("Done.")

## 3. Results: what does compression cost?
F1, AUC and agreement are averaged over the 5 folds. Latency is per image on this machine's CPU (batch of 1).

In [ ]:
g = res.groupby(["variant", "precision"])
summary = pd.DataFrame({
    "size (MB)": g.size_mb.mean().round(2),
    "F1 (%)": [f"{100 * m:.2f} ± {100 * (0 if sd != sd else sd):.2f}" for m, sd in zip(g.f1.mean(), g.f1.std())],
    "AUC (%)": (100 * g.auc.mean()).round(2),
    "missed ulcers (total)": g.missed_ulcers.sum(),
    "false alarms (total)": g.false_alarms.sum(),
    "same label as Keras (%)": (100 * g.label_agreement_vs_keras.mean()).round(2),
    "heatmap corr. vs Keras": g.heatmap_corr_vs_keras.mean().round(4),
    "CPU backend": g.cpu_backend.first(),
    **{f"CPU ms ({th} thread{'s' if th > 1 else ''})": g[f"cpu_ms_{th}thread"].mean().round(2) for th in CONFIG["THREADS"]},
}).reindex([(v, m) for v in VARIANTS for m in MODES])
summary.to_csv("tflite_summary.csv")
summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
modes = MODES; x = np.arange(len(modes)); w = 0.38
cols = {"Baseline": "#888888", SELECTED: "#b22222"}
for i, v in enumerate(VARIANTS):
    d = res[res.variant == v].groupby("precision")
    ax[0].bar(x + (i - 0.5) * w, d.size_mb.mean().loc[modes], w, color=cols[v], label=v)
    ax[1].bar(x + (i - 0.5) * w, 100 * d.f1.mean().loc[modes], w, yerr=100 * d.f1.std().loc[modes], capsize=3, color=cols[v], label=v)
    ax[2].bar(x + (i - 0.5) * w, d.cpu_ms_1thread.mean().loc[modes], w, color=cols[v], label=v)
for a, t in zip(ax, ["File size (MB)", "Test F1 (%), mean ± std over 5 folds", "CPU latency, 1 thread (ms per image)"]):
    a.set_xticks(x, modes); a.set_title(t); a.legend(fontsize=8)
ax[1].set_ylim(max(90, 100 * res.f1.min() - 2), 100.5)
plt.tight_layout(); plt.savefig("tflite_chart.png", dpi=130); plt.show()

## 4. On-device heatmap check
The INT8 heatmap comes straight from the phone-ready file. If it looks like the FP32 one, the explanation survives compression.

In [ ]:
v = SELECTED
blobs = {m: open(f"tflite/{v}_fold0_{m}.tflite", "rb").read() for m in ["FP32", "INT8"]}
te0 = np.where((kd.fold == 0) & (kd.label == 1))[0]
show = np.random.default_rng(CONFIG["SEED"]).choice(te0, 5, replace=False)
fig, ax = plt.subplots(len(show), 3, figsize=(8, 2.8 * len(show)))
for r, i in enumerate(show):
    ax[r, 0].imshow(X[i].astype(np.uint8)); ax[r, 0].set_title(kd.file[i], fontsize=8)
    for c, m in enumerate(["FP32", "INT8"], start=1):
        p, cam = run_tflite(blobs[m], X[i:i + 1])
        h = tf.image.resize(cam[..., None], (S, S)).numpy()[0, ..., 0]
        h = h / h.max() if h.max() > 0 else h
        ax[r, c].imshow(X[i].astype(np.uint8)); ax[r, c].imshow(h, cmap="jet", alpha=0.45, vmin=0, vmax=1)
        ax[r, c].set_title(f"{m}: p(ulcer) = {p[0]:.3f}", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.suptitle(f"{v}, fold-0 test images: heatmap from the TFLite file itself"); plt.tight_layout()
plt.savefig("tflite_heatmaps.png", dpi=110); plt.show()

## 5. Key numbers (printed from this run)

In [ ]:
for v in VARIANTS:
    b = summary.loc[(v, "FP32")]; q = summary.loc[(v, "INT8")]; h = summary.loc[(v, "FP16")]; d = summary.loc[(v, "INT8-dyn")]
    print(f"{v}:")
    print(f"  size FP32 {b['size (MB)']} MB -> FP16 {h['size (MB)']} MB -> INT8-dyn {d['size (MB)']} MB -> INT8 {q['size (MB)']} MB")
    print(f"  F1 FP32 {b['F1 (%)']} | FP16 {h['F1 (%)']} | INT8-dyn {d['F1 (%)']} | INT8 {q['F1 (%)']}")
    print(f"  INT8 gives the same label as the Keras model on {q['same label as Keras (%)']}% of test images; heatmap correlation {q['heatmap corr. vs Keras']}")
    print(f"  CPU latency (1 thread): FP32 {b['CPU ms (1 thread)']} ms, INT8-dyn {d['CPU ms (1 thread)']} ms, INT8 {q['CPU ms (1 thread)']} ms ({q['CPU backend']})")
print("\nFiles for the phone test are in tflite/ (fold 0, all four precisions) with tflite/thresholds_fold0.json")

## How to read the results
- **Size:** FP16 is about 2 times smaller than FP32, the INT8 versions about 3-4 times.
- **F1 / agreement:** INT8-dyn is expected to match FP32. Full-integer INT8 can lose accuracy on MobileNetV3, whose hard-swish activations
  and squeeze-and-excitation gates are known to be sensitive to post-training quantisation; whatever this run measures is the result to report.
- **Heatmap correlation:** close to 1.0 means the phone shows the same explanation as the full model.
- **Latency:** measured on Kaggle's server CPU; the Android test (next step) gives the real phone numbers.
- **CPU backend:** XNNPACK is TFLite's fast CPU library (the default on Android). If it rejects an operation of a file, the built-in kernels run it instead,
  which is correct but slower. In testing, full-integer INT8 MobileNetV3 fell back to the built-in kernels; this is reported, not hidden.

**Outputs:** `tflite_results_per_fold.csv`, `tflite_summary.csv`, `tflite_chart.png`, `tflite_heatmaps.png`, and `tflite/` (the model files for the phone).